In [3]:
import pandas as pd

In [4]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder


In [5]:
future_df = pd.read_csv("future_features_dataset.csv")

In [6]:
train_df = pd.read_csv("forecast_dataset.csv")

In [7]:
train_df

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-27,9101dx,0.0,1295.228583,3.0,0.0,510.0,True,0.24,01-27,5,1,False
2,2024-01-28,9101dx,0.0,1295.228583,3.7,0.0,516.0,True,0.24,01-28,6,1,False
3,2024-01-29,9101dx,2.0,1449.580000,5.5,0.0,492.0,True,0.24,01-29,0,1,False
4,2024-01-30,9101dx,1.0,1449.580000,6.1,0.0,192.0,True,0.24,01-30,1,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-20,sz21032,0.0,74.790000,2.3,0.0,59.0,False,0.00,11-20,3,11,False
149852,2025-11-21,sz21032,0.0,74.790000,-1.0,0.0,136.0,False,0.00,11-21,4,11,False
149853,2025-11-22,sz21032,0.0,74.790000,-4.2,0.0,243.0,False,0.00,11-22,5,11,False
149854,2025-11-23,sz21032,0.0,74.790000,-2.6,2.5,155.0,False,0.00,11-23,6,11,False


In [8]:
future_df["date"] = pd.to_datetime(future_df["date"])
train_df["date"] = pd.to_datetime(train_df["date"])

In [9]:
train_df.sort_values(by="date", inplace=True)

In [10]:
train_df.reset_index(drop=True, inplace=True)

In [11]:
train_df

,date,SKU,qty,price_per_unit,tavg,prcp,tsun,sale_active,sale_percent,mmdd,day_of_week,month,is_holiday
0,2024-01-26,9101dx,2.0,1522.660000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
1,2024-01-26,9202,2.0,277.830000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
2,2024-01-26,92018,0.0,268.461739,9.0,1.2,0.0,False,0.00,01-26,4,1,False
3,2024-01-26,sw100183,0.0,48.320000,9.0,1.2,0.0,True,0.24,01-26,4,1,False
4,2024-01-26,92016,0.0,272.180000,9.0,1.2,0.0,False,0.00,01-26,4,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
149851,2025-11-24,ga10172,0.0,671.065714,4.1,5.8,0.0,True,0.20,11-24,0,11,False
149852,2025-11-24,ga10171,0.0,531.822741,4.1,5.8,0.0,True,0.20,11-24,0,11,False
149853,2025-11-24,ga1016sf,0.0,357.635000,4.1,5.8,0.0,True,0.20,11-24,0,11,False
149854,2025-11-24,ga10263,0.0,151.543413,4.1,5.8,0.0,False,0.00,11-24,0,11,False


In [12]:
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet

# make sure date is datetime
train_df["date"] = pd.to_datetime(train_df["date"])

# sort by group + time (recommended)
train_df = train_df.sort_values(["SKU", "date"])

# create integer time index: days since first date
train_df["time_idx"] = (train_df["date"] - train_df["date"].min()).dt.days.astype("int64")


In [13]:
train_df["SKU"] = train_df["SKU"].astype("category")

# make calendar features string-based, then categorical
train_df["month"] = train_df["month"].astype(str).astype("category")
train_df["day_of_week"] = train_df["day_of_week"].astype(str).astype("category")

# bool is treated as numeric -> convert to string labels
train_df["is_holiday"] = train_df["is_holiday"].map(
    {True: "holiday", False: "no_holiday"}
).astype("category")

In [14]:
train_df.dtypes

date              datetime64[ns]
SKU                     category
qty                      float64
price_per_unit           float64
tavg                     float64
prcp                     float64
tsun                     float64
sale_active                 bool
sale_percent             float64
mmdd                      object
day_of_week             category
month                   category
is_holiday              category
time_idx                   int64
dtype: object

In [15]:
from pytorch_forecasting.data.encoders import TorchNormalizer
# create the dataset for the training period
deep_ar_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",  # or convert to an integer index
    target="qty",
    group_ids=["SKU"],
    static_categoricals=["SKU"],
    time_varying_known_reals=["price_per_unit", "tavg", "prcp", "tsun", "sale_percent"],
    time_varying_known_categoricals=["month", "day_of_week", "is_holiday"],
    time_varying_unknown_reals=["qty"],  # DeepAR uses past target values
    max_prediction_length = 100,
    max_encoder_length = 365,
    min_encoder_length = 90, # somewhere in [60, 180] (I’d start with 90)
    min_prediction_length = 30, #small for training (e.g. 1 or 30)
    target_normalizer=TorchNormalizer(method="identity", center=False),
)

In [ ]:
import lightning.pytorch as pl
from pytorch_forecasting import DeepAR
from pytorch_forecasting.metrics.distributions import NegativeBinomialDistributionLoss

import torch
torch.set_float32_matmul_precision('medium')

# create data loaders
train_dataloader = deep_ar_dataset.to_dataloader(train=True, batch_size=64, num_workers=8)

# set up the trainer
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    gradient_clip_val=0.1,
)

# configure DeepAR with a negative‑binomial likelihood
model = DeepAR.from_dataset(
    deep_ar_dataset,
    learning_rate=1e-3,
    hidden_size=64,
    rnn_layers=2,
    loss=NegativeBinomialDistributionLoss(),
)

# train the network
trainer.fit(model, train_dataloaders=train_dataloader)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                   | Type                             | Params | Mode  | FLOPs
--------------------------------------------------------------------------------------------
0 | loss                   | NegativeBinomialDistributionLoss | 0      | train | 0    
1 | logging_metrics        | ModuleList                       | 0      | train | 0    
2 | embeddings             | MultiEmbedding                   | 7.5 K  | train | 0    
3 | rnn                    | LSTM                             | 63.5 K | train | 0    
4 | distribution_projector | Linear                           | 130    | train | 0    
---------------------------------------------------------

Training: |          | 0/? [00:00<?, ?it/s]

/home/roman/eleo/eleo-mind/packages/backend/.venv/lib/python3.11/site-packages/lightning/pytorch/loops/training_epoch_loop.py:500: ReduceLROnPlateau conditioned on metric val_loss which is not available but strict is set to `False`. Skipping learning rate update.
